# 07 · Higher-Order Local Structure: 4-Gap and 5-Gap Windows

This notebook extends the earlier nearest-neighbor, ratio, and pair-correlation analysis to **higher-order local windows**.

Instead of looking only at one or two adjacent gaps, we examine short local patterns built from:

- 4-gap windows
- 5-gap windows

across three ensembles:

1. zeta-zero gaps
2. Poisson / exponential control gaps
3. finite GUE-matrix bulk gaps

## Why this matters

Standard RH numerics often focus on:

- nearest-neighbor spacing
- ratio statistics
- pair correlation

This notebook asks a less standard question:

> what short-range higher-order local structure is visible in 4-gap and 5-gap neighborhoods?

The goal is not to claim a theorem, but to build reproducible descriptors that may distinguish:

- correlated spectra (zeta / GUE)
- independent spectra (Poisson)

In [ ]:
import numpy as np
import mpmath as mp
import matplotlib.pyplot as plt

mp.mp.dps = 50
rng = np.random.default_rng(9423)

## Data sources

In [ ]:
# Zeta gaps
N = 600
zeros = [mp.zetazero(n) for n in range(1, N + 1)]
t = np.array([float(mp.im(z)) for z in zeros])
zeta_gaps = np.diff(t)

# Poisson control
poisson_gaps = rng.exponential(scale=1.0, size=len(zeta_gaps))

# Finite GUE-matrix bulk gaps
def sample_gue_bulk_spacings(matrix_size=120, n_matrices=32, bulk_fraction=0.5, rng=None):
    if rng is None:
        rng = np.random.default_rng()

    all_gaps = []
    for _ in range(n_matrices):
        real = rng.normal(size=(matrix_size, matrix_size))
        imag = rng.normal(size=(matrix_size, matrix_size))
        A = real + 1j * imag
        H = (A + A.conj().T) / 2.0
        eigvals = np.linalg.eigvalsh(H)
        eigvals = np.sort(eigvals)

        n = len(eigvals)
        keep = int(n * bulk_fraction)
        start = (n - keep) // 2
        stop = start + keep
        bulk_vals = eigvals[start:stop]
        bulk_gaps = np.diff(bulk_vals)
        all_gaps.extend(bulk_gaps.tolist())

    return np.array(all_gaps)

gue_gaps = sample_gue_bulk_spacings(matrix_size=120, n_matrices=32, bulk_fraction=0.5, rng=rng)

len(zeta_gaps), len(poisson_gaps), len(gue_gaps)

## Local normalization helper

For comparing local window shapes, we normalize each window by its own mean.
That way we focus on **shape** rather than absolute scale.

In [ ]:
def normalize_window(window):
    w = np.array(window, dtype=float)
    return w / w.mean()

## Window extraction

A 4-gap window is

\[
(g_n, g_{n+1}, g_{n+2}, g_{n+3})
\]

and similarly for a 5-gap window.

In [ ]:
def extract_windows(gaps, k):
    return np.array([gaps[i:i+k] for i in range(len(gaps) - k + 1)])

zeta_w4 = extract_windows(zeta_gaps, 4)
poisson_w4 = extract_windows(poisson_gaps, 4)
gue_w4 = extract_windows(gue_gaps, 4)

zeta_w5 = extract_windows(zeta_gaps, 5)
poisson_w5 = extract_windows(poisson_gaps, 5)
gue_w5 = extract_windows(gue_gaps, 5)

zeta_w4.shape, zeta_w5.shape, gue_w5.shape

## Normalize each local window by its own mean

In [ ]:
zeta_w4n = np.array([normalize_window(w) for w in zeta_w4])
poisson_w4n = np.array([normalize_window(w) for w in poisson_w4])
gue_w4n = np.array([normalize_window(w) for w in gue_w4])

zeta_w5n = np.array([normalize_window(w) for w in zeta_w5])
poisson_w5n = np.array([normalize_window(w) for w in poisson_w5])
gue_w5n = np.array([normalize_window(w) for w in gue_w5])

## Mean normalized 4-gap and 5-gap profiles

If there were no local position bias inside the window, the mean profile would be close to flat.

In [ ]:
def mean_profile(windows):
    return windows.mean(axis=0)

profiles = {
    "zeta_w4": mean_profile(zeta_w4n),
    "poisson_w4": mean_profile(poisson_w4n),
    "gue_w4": mean_profile(gue_w4n),
    "zeta_w5": mean_profile(zeta_w5n),
    "poisson_w5": mean_profile(poisson_w5n),
    "gue_w5": mean_profile(gue_w5n),
}
profiles

In [ ]:
x4 = np.arange(1, 5)
plt.figure(figsize=(8.2, 4.8))
plt.plot(x4, profiles["zeta_w4"], marker="o", label="zeta")
plt.plot(x4, profiles["poisson_w4"], marker="o", label="Poisson")
plt.plot(x4, profiles["gue_w4"], marker="o", label="GUE matrix")
plt.axhline(1.0, linestyle="--", linewidth=1)
plt.xlabel("position in 4-gap window")
plt.ylabel("mean normalized gap")
plt.title("Mean normalized 4-gap profile")
plt.legend()
plt.show()

In [ ]:
x5 = np.arange(1, 6)
plt.figure(figsize=(8.2, 4.8))
plt.plot(x5, profiles["zeta_w5"], marker="o", label="zeta")
plt.plot(x5, profiles["poisson_w5"], marker="o", label="Poisson")
plt.plot(x5, profiles["gue_w5"], marker="o", label="GUE matrix")
plt.axhline(1.0, linestyle="--", linewidth=1)
plt.xlabel("position in 5-gap window")
plt.ylabel("mean normalized gap")
plt.title("Mean normalized 5-gap profile")
plt.legend()
plt.show()

## Local unevenness score

For a normalized window \(w\), define

\[
U(w) = \max(w) - \min(w).
\]

This measures how uneven the local gap pattern is inside the window.

In [ ]:
def unevenness(windows):
    return windows.max(axis=1) - windows.min(axis=1)

zeta_u4 = unevenness(zeta_w4n)
poisson_u4 = unevenness(poisson_w4n)
gue_u4 = unevenness(gue_w4n)

zeta_u5 = unevenness(zeta_w5n)
poisson_u5 = unevenness(poisson_w5n)
gue_u5 = unevenness(gue_w5n)

zeta_u4.mean(), poisson_u4.mean(), gue_u4.mean()

In [ ]:
plt.figure(figsize=(8.8, 5.0))
bins = np.linspace(0, 3.0, 32)
plt.hist(zeta_u4, bins=bins, density=True, alpha=0.55, label="zeta 4-gap")
plt.hist(poisson_u4, bins=bins, density=True, alpha=0.45, label="Poisson 4-gap")
plt.hist(gue_u4, bins=bins, density=True, alpha=0.45, label="GUE 4-gap")
plt.xlabel("unevenness")
plt.ylabel("density")
plt.title("4-gap local unevenness comparison")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(8.8, 5.0))
bins = np.linspace(0, 3.5, 36)
plt.hist(zeta_u5, bins=bins, density=True, alpha=0.55, label="zeta 5-gap")
plt.hist(poisson_u5, bins=bins, density=True, alpha=0.45, label="Poisson 5-gap")
plt.hist(gue_u5, bins=bins, density=True, alpha=0.45, label="GUE 5-gap")
plt.xlabel("unevenness")
plt.ylabel("density")
plt.title("5-gap local unevenness comparison")
plt.legend()
plt.show()

## Local symmetry score

For a normalized window \(w=(w_1,\dots,w_k)\), define a reversal-symmetry score:

\[
S(w) = \frac{1}{k}\sum_{j=1}^k |w_j - w_{k+1-j}|.
\]

Smaller values mean the window is more symmetric under reversal.

In [ ]:
def reversal_symmetry_score(windows):
    rev = windows[:, ::-1]
    return np.mean(np.abs(windows - rev), axis=1)

zeta_s4 = reversal_symmetry_score(zeta_w4n)
poisson_s4 = reversal_symmetry_score(poisson_w4n)
gue_s4 = reversal_symmetry_score(gue_w4n)

zeta_s5 = reversal_symmetry_score(zeta_w5n)
poisson_s5 = reversal_symmetry_score(poisson_w5n)
gue_s5 = reversal_symmetry_score(gue_w5n)

zeta_s4.mean(), poisson_s4.mean(), gue_s4.mean()

In [ ]:
plt.figure(figsize=(8.8, 5.0))
bins = np.linspace(0, 2.0, 30)
plt.hist(zeta_s4, bins=bins, density=True, alpha=0.55, label="zeta 4-gap")
plt.hist(poisson_s4, bins=bins, density=True, alpha=0.45, label="Poisson 4-gap")
plt.hist(gue_s4, bins=bins, density=True, alpha=0.45, label="GUE 4-gap")
plt.xlabel("reversal symmetry score")
plt.ylabel("density")
plt.title("4-gap reversal-symmetry comparison")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(8.8, 5.0))
bins = np.linspace(0, 2.0, 30)
plt.hist(zeta_s5, bins=bins, density=True, alpha=0.55, label="zeta 5-gap")
plt.hist(poisson_s5, bins=bins, density=True, alpha=0.45, label="Poisson 5-gap")
plt.hist(gue_s5, bins=bins, density=True, alpha=0.45, label="GUE 5-gap")
plt.xlabel("reversal symmetry score")
plt.ylabel("density")
plt.title("5-gap reversal-symmetry comparison")
plt.legend()
plt.show()

## Window entropy score

Given a normalized window \(w\), define a local entropy-like score using
\[
p_j = \frac{w_j}{\sum_i w_i}.
\]

Then
\[
H(w) = -\sum_j p_j \log p_j.
\]

This is largest when the window is as uniform as possible.

In [ ]:
def window_entropy(windows):
    eps = 1e-12
    p = windows / windows.sum(axis=1, keepdims=True)
    return -np.sum(p * np.log(p + eps), axis=1)

zeta_h4 = window_entropy(zeta_w4n)
poisson_h4 = window_entropy(poisson_w4n)
gue_h4 = window_entropy(gue_w4n)

zeta_h5 = window_entropy(zeta_w5n)
poisson_h5 = window_entropy(poisson_w5n)
gue_h5 = window_entropy(gue_w5n)

zeta_h4.mean(), poisson_h4.mean(), gue_h4.mean()

In [ ]:
plt.figure(figsize=(8.8, 5.0))
bins = np.linspace(0.6, np.log(4), 28)
plt.hist(zeta_h4, bins=bins, density=True, alpha=0.55, label="zeta 4-gap")
plt.hist(poisson_h4, bins=bins, density=True, alpha=0.45, label="Poisson 4-gap")
plt.hist(gue_h4, bins=bins, density=True, alpha=0.45, label="GUE 4-gap")
plt.xlabel("window entropy")
plt.ylabel("density")
plt.title("4-gap entropy comparison")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(8.8, 5.0))
bins = np.linspace(0.8, np.log(5), 30)
plt.hist(zeta_h5, bins=bins, density=True, alpha=0.55, label="zeta 5-gap")
plt.hist(poisson_h5, bins=bins, density=True, alpha=0.45, label="Poisson 5-gap")
plt.hist(gue_h5, bins=bins, density=True, alpha=0.45, label="GUE 5-gap")
plt.xlabel("window entropy")
plt.ylabel("density")
plt.title("5-gap entropy comparison")
plt.legend()
plt.show()

## Example local windows

Show a few normalized windows directly for each ensemble.

In [ ]:
zeta_examples = zeta_w5n[:5]
poisson_examples = poisson_w5n[:5]
gue_examples = gue_w5n[:5]

zeta_examples, poisson_examples, gue_examples

## Mean descriptor summary

This is a compact comparison table for the higher-order window descriptors.

In [ ]:
summary = {
    "4_gap_unevenness_mean": {
        "zeta": float(zeta_u4.mean()),
        "Poisson": float(poisson_u4.mean()),
        "GUE": float(gue_u4.mean()),
    },
    "5_gap_unevenness_mean": {
        "zeta": float(zeta_u5.mean()),
        "Poisson": float(poisson_u5.mean()),
        "GUE": float(gue_u5.mean()),
    },
    "4_gap_symmetry_mean": {
        "zeta": float(zeta_s4.mean()),
        "Poisson": float(poisson_s4.mean()),
        "GUE": float(gue_s4.mean()),
    },
    "5_gap_symmetry_mean": {
        "zeta": float(zeta_s5.mean()),
        "Poisson": float(poisson_s5.mean()),
        "GUE": float(gue_s5.mean()),
    },
    "4_gap_entropy_mean": {
        "zeta": float(zeta_h4.mean()),
        "Poisson": float(poisson_h4.mean()),
        "GUE": float(gue_h4.mean()),
    },
    "5_gap_entropy_mean": {
        "zeta": float(zeta_h5.mean()),
        "Poisson": float(poisson_h5.mean()),
        "GUE": float(gue_h5.mean()),
    },
}
summary

## Notes

- These 4-gap and 5-gap descriptors are not standard canonical RH observables; they are local exploratory statistics.
- Window normalization by the local mean lets us compare **shape** rather than absolute scale.
- If zeta and GUE consistently resemble each other more than either resembles Poisson across these descriptors, that supports the idea that higher-order local structure also reflects random-matrix-style correlations.
- This notebook is a first pass; stronger versions could use:
  - larger samples
  - more careful finite-size control
  - direct multivariate comparisons instead of one descriptor at a time

## Next directions

A natural next notebook could do one of these:

1. compare 4-gap / 5-gap descriptors using scatter plots or low-dimensional embeddings  
2. estimate a properly normalized pair-correlation function as a more standard benchmark  
3. study conditional windows following unusually small or unusually large gaps  
4. build a classifier to distinguish zeta / Poisson / GUE windows from descriptor vectors